In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import os

# ==========================================
# 1. SETUP E CARREGAMENTO
# ==========================================
# Apontando para a raiz do projeto (ajuste se o notebook estiver em subpasta)
raiz_do_projeto = os.path.abspath("") 
if raiz_do_projeto not in sys.path:
    sys.path.append(raiz_do_projeto)

from src.utils.paths import ROOT
from src.utils.data_loader import load_preprocessed_data

print("⏳ Carregando textos originais...")
df_textos = load_preprocessed_data(only_valid_ids=True, columns=['id', 'text_clean'])

print("⏳ Carregando as notas do Perspective...")
caminho_notas = ROOT / "data" / "processed" / "toxicidade_perspective_COMPLETO.csv"
df_notas = pd.read_csv(caminho_notas)

print("🔄 Cruzando as bases...")
df_final = pd.merge(df_notas, df_textos, on='id', how='left')

# Mantendo apenas os posts com notas válidas para os gráficos
df_final = df_final.dropna(subset=['perspective_toxicity']).copy()
print(f"✅ Base pronta com {len(df_final)} posts válidos!\n")

# ==========================================
# 2. ESTATÍSTICAS DESCRITIVAS
# ==========================================
colunas_notas = [
    'perspective_toxicity', 'severe_toxicity', 'identity_attack', 
    'insult', 'profanity', 'threat'
]

print("-" * 50)
print("📊 ESTATÍSTICAS DOS ATRIBUTOS:")
print("-" * 50)
print(df_final[colunas_notas].describe())
print("\n")

# ==========================================
# 3. GERAÇÃO DOS GRÁFICOS
# ==========================================
# Estilo acadêmico
sns.set_theme(style="whitegrid", context="paper", font_scale=1.2)

# Gráfico 1: Barras (A Pirâmide da Agressividade)
plt.figure(figsize=(10, 5))
medias = df_final[colunas_notas].mean().sort_values(ascending=False)
sns.barplot(x=medias.values, y=medias.index, palette="mako")
plt.title('Média de Pontuação por Dimensão de Toxicidade', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Pontuação Média', fontsize=12)
plt.ylabel('Dimensão (Perspective API)', fontsize=12)
plt.tight_layout()
plt.show()

print("-" * 50)

# Gráfico 2: Boxplot (Distribuição e Outliers)
plt.figure(figsize=(12, 6))
sns.boxplot(data=df_final[colunas_notas], orient='h', palette="Set2", fliersize=1)
plt.title('Distribuição Completa das Notas e Outliers', fontsize=14, pad=15, fontweight='bold')
plt.xlabel('Pontuação (Escala de 0.0 a 1.0)', fontsize=12)
plt.ylabel('Dimensão (Perspective API)', fontsize=12)
plt.tight_layout()
plt.show()